<a href="https://colab.research.google.com/github/rohans-19/Job-Search-AI-Agent/blob/main/jobSearch2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
def fetch_jobs_adzuna(query, location, app_id, app_key):
    """
    Hits the Adzuna API and returns the top 10 job listings
    for a given role and location in India.
    """
    url = "https://api.adzuna.com/v1/api/jobs/in/search/1"

    params = {
        "app_id": app_id,
        "app_key": app_key,
        "what": query,
        "where": location,
        "results_per_page": 10
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print(f"Something went wrong — API returned status {response.status_code}")
        return []

    data = response.json()

    jobs = []
    for job in data.get("results", []):
        jobs.append({
            "title": job.get("title", "N/A"),
            "company": job["company"]["display_name"],
            "location": job["location"]["display_name"],
            "salary_min": job.get("salary_min"),
            "salary_max": job.get("salary_max"),
            "redirect_url": job.get("redirect_url", "")
        })

    return jobs

In [ ]:
app_id = "c6f6b204"
app_key = "5fd40344881248436df8a9b0198ef5c6"

jobs = fetch_jobs_adzuna("Python Developer", "Bengaluru", app_id, app_key)

df = pd.DataFrame(jobs)
df

,title,company,location,salary_min,salary_max
0,Python PHP Fullstack developer,InfoBeans,"Bangalore, Karnataka",NaN,NaN
1,"Python Developer (Apache Airflow, ETL, GCP, De...",UST,"Bangalore, Karnataka",NaN,NaN
2,AI/ML Engineer,Recro,"Bangalore, Karnataka",NaN,NaN
3,Firmware Engineer,HCLTech,"Bangalore, Karnataka",NaN,NaN
4,Python Developer,TALPRO INDIA PRIVATE LIMITED,"Madivala, Bangalore",1200000.0,1200000.0
5,Python Developer,WebGo Software Labs,"Bangalore, Karnataka",500000.0,700000.0
6,Python Developer,viraaj hr solutions,"Bangalore, Karnataka",1500000.0,3000000.0
7,Python Developer,ALIQAN Technologies,"Bangalore, Karnataka",80000.0,170000.0
8,Python Developer,GrowthFalcons,"Bangalore, Karnataka",0.0,600000.0
9,Python Developer,Sagacious Programming & Development Pvt Ltd,"Bangalore, Karnataka",500000.0,1200000.0


In [ ]:
def job_chatbot_api(app_id, app_key):
    """
    A simple terminal chatbot — ask it for a role and city,
    and it'll pull the top matches straight from Adzuna.
    """
    print("AI Career Assistant — India")
    print("Type 'exit' at any time to quit\n")

    while True:
        role = input("What role are you looking for? ").strip()
        if role.lower() == "exit":
            print("Good luck with your job search!")
            break

        location = input("Which city? ").strip()

        jobs = fetch_jobs_adzuna(role, location, app_id, app_key)

        if not jobs:
            print("Couldn't find anything for that. Try a different role or city.\n")
            continue

        print(f"\nHere are the top {min(5, len(jobs))} results:\n")
        for i, job in enumerate(jobs[:5], 1):
            salary = (
                f"₹{job['salary_min']:,.0f} – ₹{job['salary_max']:,.0f}"
                if job["salary_min"] and job["salary_max"]
                else "Not disclosed"
            )
            print(f"{i}. {job['title']}")
            print(f"   Company  : {job['company']}")
            print(f"   Location : {job['location']}")
            print(f"   Salary   : {salary}")
            if job.get("redirect_url"):
                print(f"   Apply    : {job['redirect_url']}")
            print("   " + "-" * 40)
        print()


job_chatbot_api(app_id, app_key)

In [ ]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup


def scrape_naukri(job_role, job_location, num_pages=2):
    """
    Scrapes Naukri.com for job listings matching the given role and city.
    Returns a list of dicts with title, company, location, and salary info.

    Note: This relies on Naukri's HTML structure — if it breaks, their site
    likely changed layout and the selectors below need updating.
    """
    all_jobs = []
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    }

    formatted_role = job_role.lower().replace(" ", "-")
    formatted_location = job_location.lower().replace(" ", "-")

    for page_num in range(1, num_pages + 1):
        print(f"Scraping page {page_num} for '{job_role}' in '{job_location}'...")

        url = f"https://www.naukri.com/{formatted_role}-jobs-in-{formatted_location}-{page_num}"

        try:
            response = requests.get(url, headers=headers, timeout=15)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, "html.parser")

            job_elements = soup.find_all("article", class_="jobTuple bgWhite br4 mb-8")

            if not job_elements:
                print(f"No listings found on page {page_num}. Stopping early.")
                break

            for job in job_elements:
                title_elem = job.find("a", class_="title fw500 ellipsis")
                title = title_elem.get("title", "N/A") if title_elem else "N/A"

                company_elem = job.find("a", class_="subTitle ellipsis fleft")
                company = company_elem.text.strip() if company_elem else "N/A"

                location_elem = job.find("span", class_="ellipsis fleft locWdth")
                location = location_elem.text.strip() if location_elem else "N/A"

                salary_elem = job.find("span", class_="ellipsis fleft salary")
                salary_raw = salary_elem.text.strip() if salary_elem else "N/A"

                salary_min, salary_max = None, None
                if salary_raw and salary_raw != "N/A":
                    try:
                        clean = salary_raw.replace("P.A.", "").replace(" ", "").replace(",", "")
                        if "-" in clean:
                            lo, hi = clean.split("-", 1)
                            salary_min = float("".join(filter(str.isdigit, lo.replace("Lacs", "00000").replace("Crore", "0000000"))))
                            salary_max = float("".join(filter(str.isdigit, hi.replace("Lacs", "00000").replace("Crore", "0000000"))))
                        else:
                            salary_min = float("".join(filter(str.isdigit, clean.replace("Lacs", "00000").replace("Crore", "0000000"))))
                            salary_max = salary_min
                    except ValueError:
                        pass

                all_jobs.append({
                    "title": title,
                    "company": company,
                    "location": location,
                    "salary_min": salary_min,
                    "salary_max": salary_max,
                    "salary_raw": salary_raw
                })

        except requests.exceptions.RequestException as e:
            print(f"Network error on page {page_num}: {e}")
            break
        except Exception as e:
            print(f"Unexpected error on page {page_num}: {e}")
            break

        time.sleep(2)

    return all_jobs


job_role = input("Enter job role (e.g. 'Python Developer'): ")
job_location = input("Enter city (e.g. 'Bengaluru'): ")

print(f"\nScraping Naukri.com for '{job_role}' in '{job_location}'...\n")
naukri_jobs = scrape_naukri(job_role, job_location, num_pages=2)

df_naukri = pd.DataFrame(naukri_jobs)

if not df_naukri.empty:
    print(df_naukri.head(10).to_markdown(index=False))
else:
    print("No jobs found or something went wrong during scraping.")

Enter job role (e.g., 'Python Developer'): Python Developer
Enter job location (e.g., 'Bengaluru'): Bengaluru

--- Starting Naukri.com scraping for 'Python Developer' in 'Bengaluru' for 2 page(s) using HTML parsing. ---
Scraping page 1 for 'Python Developer' in 'Bengaluru' from Naukri.com...
No job listings found on page 1. Ending scrape.

--- First 10 job listings from Naukri.com ---
No jobs found or an error occurred during scraping.
